# 🧠 Brain CT Hemorrhage Detection
**Model:** EfficientNet-B3 + GradCAM  
**Task:** Binary classification — `hemorrhage` vs `normal`  
**Dataset:** `drive/MyDrive/processed_dataset/{train,val,test}/{hemorrhage,normal}`

---
### 🔧 Steps:
1. Run **Cell 1** — Mount Drive & install deps
2. Run **Cell 2** — Upload/verify the training script
3. Run **Cell 3** — Train the model (40 epochs, ~30–45 min on T4)
4. Run **Cell 4** — Inference on a single image


In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Install extra deps (sklearn + seaborn for metrics)
!pip install -q scikit-learn seaborn

# Verify GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Verify dataset structure
import os
DATA_ROOT = '/content/drive/MyDrive/processed_dataset'
for split in ['train', 'val', 'test']:
    for cls in ['hemorrhage', 'normal']:
        path = os.path.join(DATA_ROOT, split, cls)
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f'  ✅ {split}/{cls}: {count} images')
        else:
            print(f'  ❌ MISSING: {path}')

In [ ]:
# ── Cell 2: Upload the training script ────────────────────────────────────
# Option A — if you uploaded brain_hemorrhage_model.py to Colab files:
# (already in /content/ from the Files panel upload)

# Option B — copy from Drive (if you saved it there):
# !cp '/content/drive/MyDrive/brain_hemorrhage_model.py' /content/

# Option C — paste inline (advanced)
# Verify the file is present:
!ls -lh /content/brain_hemorrhage_model.py

In [ ]:
# ── Cell 3: Train ──────────────────────────────────────────────────────────
# This runs the full training pipeline.
# Outputs saved to: /content/drive/MyDrive/hemorrhage_model_outputs/
%run /content/brain_hemorrhage_model.py

In [ ]:
# ── Cell 4: Inference on a single image ───────────────────────────────────
import torch
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# ── Re-import model class (run this after training)
import importlib.util, sys
spec = importlib.util.spec_from_file_location('model_module', '/content/brain_hemorrhage_model.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT     = '/content/drive/MyDrive/hemorrhage_model_outputs/best_model.pth'
IMG_PATH = '/path/to/your/test_image.png'   # ← change this

# Load model
ckpt  = torch.load(CKPT, map_location=DEVICE)
model = mod.HemorrhageDetector(num_classes=2).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

idx_to_class = {v: k for k, v in ckpt['class_to_idx'].items()}

# Preprocess
tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
img   = Image.open(IMG_PATH).convert('RGB')
tensor = tfm(img)

# GradCAM
target_layer = model.features[-1]
gc = mod.GradCAM(model, target_layer)
cam, pred_idx, probs = gc.generate(tensor)
orig, heatmap, overlay = mod.overlay_gradcam(tensor, cam)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(orig);    axes[0].set_title('Original CT');      axes[0].axis('off')
axes[1].imshow(heatmap); axes[1].set_title('GradCAM Heatmap');  axes[1].axis('off')
axes[2].imshow(overlay); axes[2].set_title(
    f'Prediction: {idx_to_class[pred_idx]}\nConfidence: {probs[pred_idx]*100:.1f}%'
); axes[2].axis('off')
plt.suptitle('Brain Hemorrhage Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n🔍 Prediction  : {idx_to_class[pred_idx]}')
print(f'   Hemorrhage  : {probs[0]*100:.2f}%')
print(f'   Normal      : {probs[1]*100:.2f}%')